In [ ]:
import pandas as pd
import requests
import time
import re
from tqdm import tqdm

# ==========================================
# CONFIGURATION
# ==========================================
# Replace this with your actual TMDB API key (v3 auth)
TMDB_API_KEY = "92a48f2ce3dea72aecb5346f1c502b27"

INPUT_CSV = "/Users/adonisgeoffmacias/Documents/GitHub/data-trio-project/DMW/Data Scrapping RESULTS/user_ratings.csv"
OUTPUT_CSV = "user_ratings_enriched2.csv"

# ==========================================
# FUNCTIONS
# ==========================================
def get_genre_mapping(api_key):
    """Fetches the official TMDB list of movie genres to map IDs to names."""
    url = f"https://api.themoviedb.org/3/genre/movie/list?api_key={api_key}&language=en-US"
    response = requests.get(url)
    if response.status_code == 200:
        genres = response.json().get('genres', [])
        return {g['id']: g['name'] for g in genres}
    print("Warning: Could not fetch genre mapping from TMDB.")
    return {}

def fetch_movie_data(slug, api_key, genre_mapping):
    """Searches TMDB for a movie based on the Letterboxd slug."""
    # Letterboxd slugs often look like 'the-super-mario-galaxy-movie' or 'michael-2026'
    # 1. Extract year if it exists at the end of the slug
    match = re.search(r'-(\d{4})$', str(slug))
    year = match.group(1) if match else None
    
    # 2. Clean the title for searching (remove the year and replace hyphens with spaces)
    clean_title = re.sub(r'-\d{4}$', '', str(slug)).replace('-', ' ')
    
    # 3. Build the search query
    url = f"https://api.themoviedb.org/3/search/movie?api_key={api_key}&query={clean_title}"
    if year:
        url += f"&primary_release_year={year}"
        
    try:
        response = requests.get(url)
        if response.status_code == 200:
            results = response.json().get('results', [])
            if results:
                # Take the top search result
                movie = results[0]
                tmdb_id = movie.get('id')
                title = movie.get('title')
                genre_ids = movie.get('genre_ids', [])
                
                # Map genre IDs to their string names
                genre_names = [genre_mapping.get(gid) for gid in genre_ids if gid in genre_mapping]
                
                return {
                    'film_slug': slug,
                    'tmdb_id': tmdb_id,
                    'proper_title': title,
                    'genres_list': genre_names
                }
    except Exception as e:
        print(f"Error fetching data for slug '{slug}': {e}")
        
    # Return empty/null values if not found or if there's an error
    return {
        'film_slug': slug,
        'tmdb_id': None,
        'proper_title': None,
        'genres_list': []
    }

# ==========================================
# MAIN SCRIPT
# ==========================================
def main():
    print(f"Loading data from {INPUT_CSV}...")
    df = pd.read_csv(INPUT_CSV)
    
    # Get unique slugs so we only search TMDB once per movie
    unique_slugs = df['film_slug'].dropna().unique()
    print(f"Found {len(unique_slugs)} unique movies to fetch.")
    
    print("Fetching TMDB genre list...")
    genre_mapping = get_genre_mapping(TMDB_API_KEY)
    
    movie_details = []
    
    print("Fetching movie details from TMDB (this may take a moment)...")
    for i, slug in enumerate(unique_slugs):
        details = fetch_movie_data(slug, TMDB_API_KEY, genre_mapping)
        movie_details.append(details)
        
        # Progress tracker
        if (i + 1) % 50 == 0:
            print(f"Processed {i + 1} / {len(unique_slugs)} movies...")
    
    # tqdm creates a progress bar with an ETA automatically!
    for slug in tqdm(unique_slugs, desc="Fetching TMDB Data", unit="movie"):
        details = fetch_movie_data(slug, TMDB_API_KEY, genre_mapping)
        movie_details.append(details)
            
        # Add a tiny sleep to respect TMDB's API rate limits (approx 40-50 req/sec)
        time.sleep(0.05)
        
    # Convert the list of dictionaries into a DataFrame
    details_df = pd.DataFrame(movie_details)
    
    print("Formatting genres into separate columns...")
    # Expand the 'genres_list' into up to 5 separate columns
    for i in range(5):
        col_name = f'genre_{i+1}'
        details_df[col_name] = details_df['genres_list'].apply(
            lambda x: x[i] if isinstance(x, list) and len(x) > i else None
        )
        
    # Drop the temporary list column
    details_df = details_df.drop(columns=['genres_list'])
    
    print("Merging data back together...")
    # Merge the fetched TMDB details back into the original dataframe based on film_slug
    final_df = pd.merge(df, details_df, on='film_slug', how='left')
    
    # Save the result to a new CSV file
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Success! Data has been saved to {OUTPUT_CSV}")

# Execute the script
if __name__ == "__main__":
    main()